In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import os
from dotenv import load_dotenv
import pandas as pd

In [3]:
load_dotenv()
URL = os.getenv("SUPABASE_URL")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")

In [4]:
# 1. Load Preprocessed Data
cloud_path = f"{URL}/storage/v1/object/public/{BUCKET_NAME}/ulasan_ikd_preprocessed.csv"
print(f"Loading data from Supabase: {cloud_path}")

df = pd.read_csv(cloud_path)

# [KOREKSI FATAL]: Kembalikan NaN menjadi string kosong khusus untuk kolom ablasi
df = df.fillna("")

print("Data loaded successfully. Shape:", df.shape)

Loading data from Supabase: https://ffygtotdlnxhscukpmpu.supabase.co/storage/v1/object/public/dataset-ikd/ulasan_ikd_preprocessed.csv
Data loaded successfully. Shape: (34902, 7)


In [3]:
# 2. Setup Variables
y = df['label']
features = ['text_baseline', 'text_standard', 'text_custom']

print("--- MEMULAI ABLATION STUDY (DEFAULT SVM) ---\n")

best_score = 0
best_dataset = ""

# 3. Looping Uji Ablasi
for col in features:
    # Stratify=y memastikan rasio kelas Positif/Negatif seimbang di Train dan Test
    X_train, X_test, y_train, y_test = train_test_split(df[col], y, test_size=0.2, random_state=42, stratify=y)
    
    # Vectorization (TF-IDF)
    tfidf = TfidfVectorizer()
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)
    
    # Model Training (Default Linear SVM)
    svm = SVC(kernel='linear', random_state=42)
    svm.fit(X_train_vec, y_train)
    
    # Prediction & Evaluation
    y_pred = svm.predict(X_test_vec)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f"Dataset : {col}")
    print(f"Macro F1: {macro_f1:.4f}")
    print("-" * 30)
    
    if macro_f1 > best_score:
        best_score = macro_f1
        best_dataset = col

print(f"\nPemenang Uji Ablasi adalah {best_dataset.upper()} dengan skor {best_score:.4f}")
print("Gunakan dataset ini untuk tahapan Optuna dan LSTM selanjutnya.")

--- MEMULAI ABLATION STUDY (DEFAULT SVM) ---

Dataset : text_baseline
Macro F1: 0.9269
------------------------------
Dataset : text_standard
Macro F1: 0.9165
------------------------------
Dataset : text_custom
Macro F1: 0.9209
------------------------------

Pemenang Uji Ablasi adalah TEXT_BASELINE dengan skor 0.9269
Gunakan dataset ini untuk tahapan Optuna dan LSTM selanjutnya.


Pengujian ablasi preprocessing menunjukkan hasil yang berlawanan dengan asumsi umum pemrosesan teks tradisional. Dataset text_baseline (tanpa penghapusan stopword) memperoleh Macro F1-Score tertinggi sebesar 0.9269. Penggunaan kamus stopword standar (text_standard) justru mendegradasi performa model ke 0.9165 karena hilangnya kata-kata pembawa konteks dan negasi. Meskipun modifikasi penghapusan stopword (text_custom) berhasil memulihkan sebagian performa menjadi 0.9209, skor tersebut tetap tidak dapat melampaui baseline. Hal ini membuktikan bahwa pada teks ulasan berukuran pendek, algoritma TF-IDF secara inheren telah mampu menangani noise dari kata-kata umum (melalui pembobotan Inverse Document Frequency) dengan lebih baik daripada penghapusan kamus statis. Berdasarkan temuan empiris ini, penelitian dilanjutkan menggunakan text_baseline.